# Cumulative Spend Distribution Analysis

This notebook characterizes the underlying cumulative spend-position distribution that feeds the later Beta CDF modeling work. It should be read before `clustered_curve_beta_cdf_model.ipynb`: this page explains the empirical shape of `CUMULATIVEBURNPCT` as a function of `ELAPSEDPCT`, while the later notebook evaluates predictive reference curves against held-out items.

## Dataset and Overall Character

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_ccd.csv |
| Cluster curve rows | 316 |
| Items | 29 |
| Train rows used for distribution fitting | 205 |
| Train items | 20 |
| Mean cumulative spend pct | 61.76% |
| Median cumulative spend pct | 70.19% |
| Std dev cumulative spend pct | 33.11% |
| Share at completed edge near 100% | 10.13% |
| Share at zero edge near 0% | 2.53% |
| Pearson corr elapsed vs cumulative spend | 0.8621 |
| Spearman corr elapsed vs cumulative spend | 0.8866 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Marginal Distribution of Cumulative Spend

The marginal distribution is bounded between 0 and 1 and is strongly affected by repeated observations from the same item curve. It is not a standalone time-free distribution: a point at 90% elapsed and a point at 20% elapsed should not be expected to have the same cumulative spend behavior. The visible pile-up near 100% is expected because every completed item contributes a final cluster at full cumulative spend.

| Quantile | Cumulative spend pct |
| --- | --- |
| p01 | 0.00% |
| p05 | 3.29% |
| p10 | 9.17% |
| p25 | 33.39% |
| p50 | 70.19% |
| p75 | 92.49% |
| p90 | 99.96% |
| p95 | 100.00% |
| p99 | 100.00% |



## Joint Distribution With Elapsed Percent

The joint view is the core characterization. The distribution is bounded, monotonic in expectation, heteroscedastic, and edge-inflated near completion. The diagonal line is the pure linear spend reference; density above the line is front-loaded spend, while density below it is back-loaded or delayed spend.



## Conditional Distribution by Elapsed Bucket

The table and band plot summarize `CUMULATIVEBURNPCT | ELAPSEDPCT bucket`. The widening and narrowing of the percentile bands matters more than the overall histogram because the production question is where an item sits relative to expected cumulative spend at its current elapsed position.

| Elapsed bucket | Rows | Mean | Median | P10 | P25 | P75 | P90 | IQR | Skew |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0.0-0.1 | 17 | 8.66% | 3.58% | 0.00% | 0.00% | 12.39% | 17.97% | 12.39% | 2.585 |
| 0.1-0.2 | 14 | 25.48% | 19.00% | 8.19% | 10.53% | 35.65% | 43.47% | 25.12% | 1.815 |
| 0.2-0.3 | 22 | 31.22% | 27.73% | 5.12% | 16.03% | 42.54% | 51.89% | 26.51% | 0.989 |
| 0.3-0.4 | 21 | 52.03% | 50.86% | 28.17% | 37.73% | 64.77% | 74.42% | 27.04% | 0.304 |
| 0.4-0.5 | 21 | 53.36% | 57.41% | 27.36% | 36.24% | 71.67% | 82.73% | 35.43% | -0.373 |
| 0.5-0.6 | 19 | 67.83% | 71.44% | 47.28% | 55.93% | 79.30% | 95.46% | 23.36% | -0.356 |
| 0.6-0.7 | 28 | 74.54% | 77.84% | 55.31% | 64.97% | 85.56% | 97.32% | 20.59% | -0.669 |
| 0.7-0.8 | 16 | 84.55% | 85.97% | 74.11% | 80.77% | 89.29% | 91.46% | 8.52% | -0.296 |
| 0.8-0.9 | 13 | 92.05% | 94.07% | 85.43% | 89.50% | 95.04% | 96.82% | 5.54% | -0.901 |
| 0.9-1.0 | 34 | 97.90% | 100.00% | 90.42% | 98.54% | 100.00% | 100.00% | 1.46% | -1.930 |

## Percentile Bands and Reference Curves

The empirical median curve is generally above the linear reference for much of the item life, which means these clustered payment curves are often front-loaded relative to time. The Beta CDF and anchored polynomial are compact parameterizations of that empirical shape; the later model notebook evaluates the Beta CDF approach more formally.



## Bucketed Density View

This view shows the full shape within each elapsed bucket rather than only percentiles. It makes the conditional spread visible: some elapsed regions are broad and multi-modal because items can receive lumpy clustered payments at different points in their lifecycle.



## Curve Parameterizations

| Model | Alpha / coefficients | Beta | MAE | RMSE | Bias | Clip share | Monotonic violations |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Beta CDF | 0.868050 | 1.211026 | 0.1164 | 0.1660 | 0.0023 |  |  |
| Anchored polynomial degree 3 | 1.003292, 0.601254, -0.255298 |  | 0.1164 | 0.1658 | 0.0010 | 0.0060 | 0 |
| Anchored polynomial degree 4 | 1.004068, 0.580833, -0.162114, -0.098961 |  | 0.1164 | 0.1658 | 0.0008 | 0.0060 | 0 |

The Beta CDF is a useful production candidate because it is naturally bounded and monotone. The anchored polynomial is useful as a flexible descriptive reference, but it is less constrained structurally and should be treated as diagnostic unless monotonicity and stability are explicitly checked.

## Residual Distribution Around Reference Curves

Residuals are `actual cumulative pct - expected cumulative pct`. A good reference curve centers this distribution closer to zero and reduces asymmetric bias. The residual plot shows why the cumulative spend model should not be purely linear: the linear reference leaves a larger systematic position offset where the empirical spend curve is front-loaded.



## Stratification by Duration

Duration buckets have visibly different median cumulative spend curves. This supports the later duration-bucket Beta CDF model: the expected curve is not fully universal across short and long items.



## Stratification by Cluster Count

Cluster count is another proxy for payment cadence and lumpiness. Items with fewer clusters tend to show coarser jumps, while higher-cluster items provide smoother cumulative curves.



## Interpretation

The cumulative spend distribution is best understood as a conditional bounded distribution, not as a single ordinary marginal distribution. Its important properties are:

- Bounded support at `[0, 1]`, with a structural edge at 100% from final clusters.
- Strong positive relationship between elapsed percent and cumulative spend percent.
- A median curve that is not purely linear, indicating systematic front-loading in the clustered payment data.
- Wide conditional dispersion, especially in the middle of the lifecycle, caused by lumpy payment postings and differing payment cadence.
- Meaningful stratification by item duration and cluster count.

This motivates the next-stage notebook: fit expected-position curves and evaluate whether Beta CDF parameterizations outperform a transparent linear reference.